# 05 — Implement Otsu

Threshold finite NDWI values and make invalid-pixel behavior explicit.

Run this lab in the **dev-platform** workspace on Minikube or Kubernetes, using the `eoepca/pde-code-server:amd64` image. Select its preinstalled **Bash** kernel and use **Shift+Enter**, reading each explanation before executing its cell. No terminal or kernel installation is needed. Stop at an unexpected error and fix it before continuing. The course environment is provisioned by workspace startup.

The kernel keeps environment variables and the working directory between cells. The first cell finds the course checkout and activates its locked environment. Command groups run in a subshell with `set -euo pipefail` so a failing command stops that cell without killing the kernel. Scratch results stay in `.notebook-runs/05-implement-otsu/`; rerunning a cell replaces only its lab outputs.

[Lesson](../modules/05-implement-otsu/lesson.md) · [Exercise](../modules/05-implement-otsu/exercise.md)


## Prepare the notebook workspace

Run this cell after every kernel restart. It clears inherited `CWLTOOL_OPTIONS` so a Coder workspace configured for Podman can still run the explicitly local exercises. It does not change your cluster or publish anything.


In [ ]:
lab_start_dir="$PWD"
while [[ ! -f reference/waterbodies.cwl || ! -f pyproject.toml ]]; do
    if [[ "$PWD" == / ]]; then
        cd "$lab_start_dir"
        break
    fi
    cd ..
done
export WORKSPACE="$PWD"
if [[ -x "$WORKSPACE/.venv/bin/python" && -f "$WORKSPACE/reference/waterbodies.cwl" ]]; then
    source "$WORKSPACE/.venv/bin/activate"
    export PYTHONPATH="$WORKSPACE${PYTHONPATH:+:$PYTHONPATH}"
    export CWLTOOL_OPTIONS=
    export MPLBACKEND=Agg
    export LAB_DIR="$WORKSPACE/.notebook-runs/05-implement-otsu"
    mkdir -p "$LAB_DIR"
    cd "$LAB_DIR"
    python --version
    waterbodies --help
else
    printf 'Open this notebook inside the course checkout. Ask the instructor to check workspace environment provisioning.\n' >&2
    false
fi


## Prepare NDWI and classify it

This lab recreates both crops and NDWI. Otsu estimates a threshold from finite values; classification uses strict greater-than comparison. Predict the class count for each half of the 6×6 AOI.


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
from pathlib import Path
from scripts.fixtures import create_fixture
create_fixture(Path('source'))
print('Created an 8 x 8 STAC scene with green and NIR assets under source/')
PYTHON

waterbodies crop --input-item source --aoi "1,1,7,7" --epsg EPSG:4326 --band green
waterbodies crop --input-item source --aoi "1,1,7,7" --epsg EPSG:4326 --band nir
waterbodies norm-diff --rasters crop_green.tif --rasters crop_nir.tif
waterbodies otsu --raster norm_diff.tif
)


## Verify the output mask

The binary mask uses uint8, values 0 and 1, and no nodata sentinel. A class count verifies the scientific result in addition to the file format.


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
import numpy as np
import rasterio
with rasterio.open('otsu.tif') as raster:
    values, counts = np.unique(raster.read(1), return_counts=True)
    print('dtype:', raster.dtypes, 'nodata:', raster.nodata)
    print('Classes and counts:', list(zip(values.tolist(), counts.tolist(), strict=True)))
    assert raster.dtypes == ('uint8',)
    assert raster.nodata is None
    np.testing.assert_array_equal(values, [0, 1])
    np.testing.assert_array_equal(counts, [18, 18])
PYTHON
)


## Explore invalid values

Predict the classes assigned to NaN and infinity. Test the public threshold operation, then test the caller-visible error when no finite population exists.


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
import numpy as np
from waterbodies.otsu_impl import threshold
pixels = np.array([-0.5, 0.5, np.nan, np.inf], dtype=np.float32)
mask = threshold(pixels)
print('Input:', pixels, 'Mask:', mask)
np.testing.assert_array_equal(mask, [0, 1, 0, 0])
try:
    threshold(np.array([np.nan, np.inf], dtype=np.float32))
except ValueError as error:
    print('Expected rejection:', error)
else:
    raise AssertionError('An entirely nonfinite population must be rejected')
PYTHON
python -m pytest "$WORKSPACE/tests/tools/test_processing.py" -k otsu
)


## Record your findings

**Expected outcome:** The default mask is uint8 with 18 pixels per class; nonfinite pixels are non-water and an entirely invalid population fails.

Add your observations to a Markdown cell: what you predicted, what the command produced, and which contract declaration explains it. Restart the kernel and run all cells with the default settings to check reproducibility.

Compare [the solution](../modules/05-implement-otsu/solution.md) after the activity. Return to [the lab index](README.md).
